<a href="https://colab.research.google.com/github/mshinno26/UnderstandingAI/blob/main/neural_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the code from https://sirupsen.com/napkin/neural-net, use this to evolve the model with the suggested exercises in the article

Imports

In [1]:
from google.colab import drive
import torch
import torch.nn.functional as F
import librosa
import os
import numpy as np
from sklearn.model_selection import train_test_split

Google Drive mounting

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Universal constants

In [ ]:
data_dir = "/content/drive/My Drive/Grade 12/AI/Numbers/Dataset"

n_mfcc = 13 # number of MFCC coefficients
max_len = 50 # number of time steps (change to match ideal sound file length)
num_classes = 10 # 0-9
input_size = n_mfcc * max_len # size of tensors

Class for MFCC conversion

In [ ]:
class Converter:
  def __init__(self, n_mfcc):
    self._n_mfcc = n_mfcc

  def set_path(self, path):
    # Load audio
    self._audio, self._sr = librosa.load(path, sr=None)

  def convert(self, max_len):
    self._mfcc = librosa.feature.mfcc(y=self._audio, sr=self._sr, n_mfcc=self._n_mfcc)
    self.pad(max_len)
    return self.get_flat()

  def pad(self, max_len):
    # pad or truncate vectors to make them all the same length, regardless of audio file length
    length = self._mfcc.shape[1]
    if length < max_len:
        pad_size = max_len - length
        self._mfcc = np.pad(self._mfcc, ((0, 0), (0, pad_size)))
    else:
        self._mfcc = self._mfcc[:, :max_len]

  def get_flat(self):
    # Flatten from 2D to 1D vector
    return self._mfcc.flatten()

Convert sound files to MFCC vectors & store them

In [ ]:
X = []
y = []

converter = Converter(n_mfcc)

# loop through audio files in each number's folder in the directory containing training set
for label in range(num_classes):
    folder = os.path.join(data_dir, str(label))

    for file in os.listdir(folder):
        path = os.path.join(folder, file)

        converter.set_path(path)
        mfcc = converter.convert(max_len)

        X.append(mfcc)
        y.append(label)

/tmp/ipykernel_1152/2146966364.py:7: UserWarning: PySoundFile failed. Trying audioread instead.
  self._audio, self._sr = librosa.load(path, sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_1152/2146966364.py:7: UserWarning: PySoundFile failed. Trying audioread instead.
  self._audio, self._sr = librosa.load(path, sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_1152/2146966364.py:7: UserWarning: PySoundFile failed. Trying audioread instead.
  self._audio, self._sr = librosa.load(path, sr=None)
/usr/local/l

Split training & testing sets, convert to tensors

In [ ]:
# Convert to numpy array so scikit can work with them
X = np.array(X)
y = np.array(y)

# Normalize MFCC vectors (values can differ a lot, which wouldn't be good I guess?)
X = (X - X.mean()) / X.std()

# Split training & testing; "stratify=y" ensures that there is an equal number of each class in the sets; for example, 20% of zeroes and 20% of ones, as opposed to a random 20% of all numbers
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Convert from arrays to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

# Normalize datatypes
X_train = X_train.float()
X_test = X_test.float()
y_train = y_train.long()
y_test = y_test.long()

print("X training shape:", X_train.shape)
print("X testing shape:", X_test.shape)
print("y training shape:", y_train.shape)
print("y testing shape:", y_test.shape)

X training shape: torch.Size([64, 650])
X testing shape: torch.Size([16, 650])
y training shape: torch.Size([64])
y testing shape: torch.Size([16])


Neural net class

In [ ]:
class FCNN:
  def __init__(self, input_size, num_classes, hidden_size = 64, learning_rate = 0.001):
    self._input_size = input_size
    self._num_classes = num_classes
    self._hidden_size = hidden_size
    self._learning_rate = learning_rate

    # Initialize Weights for 2 Layers
    # Layer 1: 650 length flattened vector -> 64 hidden neurons
    self._w1 = torch.randn(self._input_size, self._hidden_size) * 0.01
    self._w1.requires_grad_()
    self._b1 = torch.zeros(self._hidden_size, requires_grad=True)

    # Layer 2: 64 hidden neurons -> 10 probabilities
    self._w2 = torch.randn(self._hidden_size, self._num_classes) * 0.01
    self._w2.requires_grad_()
    self._b2 = torch.zeros(self._num_classes, requires_grad=True)

  def classify(self, x):
    # Layer 1 with ReLU activation (the non-linear part)
    hidden = torch.relu(torch.matmul(x, self._w1) + self._b1)
    # Layer 2 (output)
    output = torch.matmul(hidden, self._w2) + self._b2
    return output

  def train(self, X_train, y_train):
    # Training Loop
    learning_rate = 0.1
    for epoch in range(5000):
      preds = self.classify(X_train)
      loss = F.cross_entropy(preds, y_train)

      loss.backward()

      with torch.no_grad():
        for param in [self._w1, self._b1, self._w2, self._b2]:
          param -= learning_rate * param.grad
          param.grad.zero_()

  def test(self, X_test, y_test):
    with torch.no_grad():
      out = self.classify(X_test)
      print(out)

      predictions = torch.argmax(out, dim = 1)

      accuracy = (predictions == y_test).float().mean()
      print(predictions)
      print(y_test)

    return accuracy.item()

Train and test with existing dataset

In [ ]:
fcnn = FCNN(input_size, num_classes)
fcnn.train(X_train, y_train)
accuracy = fcnn.test(X_test, y_test)
print("Accuracy: ", accuracy)

tensor([[-2.1675,  0.0663,  0.3016, -0.5413, -0.0956,  0.2994,  0.2834,  0.5824,
          0.9251,  0.1595],
        [ 1.4013,  0.7891,  0.2700,  0.7343,  0.0682, -0.2901, -1.5913, -0.4345,
         -1.1602, -0.0557],
        [-2.8228,  0.3976,  0.4589, -1.0367,  0.0780,  0.6635,  0.7207,  0.9299,
         -0.8394,  1.0800],
        [-0.5817,  0.2518,  0.3674, -0.1682,  0.3197,  0.4423,  0.0497,  0.1029,
         -1.1588,  0.1013],
        [-0.3695,  0.7148,  0.1985,  0.3328, -0.4711, -0.5360, -1.5121,  0.1405,
          1.1458,  0.1635],
        [ 1.2166,  0.6592,  0.2338,  0.7370, -0.0056, -0.3400, -1.5513, -0.4151,
         -0.5145, -0.2383],
        [-4.5513,  0.0634,  0.4452, -1.5890, -0.1052,  0.8256,  1.4592,  1.4226,
          0.5891,  1.1331],
        [ 0.9134,  0.4761,  0.1505,  0.7977, -0.2227, -0.5357, -1.6497, -0.3782,
          0.8907, -0.5609],
        [-3.5122, -0.2092,  0.3508, -1.1035, -0.0347,  0.6677,  1.1824,  0.9705,
          1.1747,  0.3257],
        [-0.7871,  

Predict a singular sound

In [ ]:
single_folder_path =
path = ""

for file in os.listdir(single_folder_path):
  path = os.path.join(single_folder_path, file)

converter.set_path(path)
mfcc = converter.convert(max_len)

X = (mfcc - mfcc.mean()) / mfcc.std()
X = np.array(X)
X = torch.tensor(X, dtype=torch.float32)

print("Predicted label: ", fcnn.classify(X))